In [1]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

## House price regression

Trying a Keras regression model on the King County house sales data.

### What I tried
- Start with all rows and a small dense network
- Train one more time after dropping the top 2% most expensive houses

In [2]:
tf.keras.utils.set_random_seed(42)
np.random.seed(42)
random.seed(42)

data_candidates = [
    Path("projects/house-price-regression/data/kc_house_data.csv"),
    Path("data/kc_house_data.csv"),
    Path("../data/kc_house_data.csv"),
]

for candidate in data_candidates:
    if candidate.exists():
        data_path = candidate
        break
else:
    raise FileNotFoundError(
        "Couldn't find kc_house_data.csv from the repo root, project folder, or codes folder."
    )

print(f"Using data from: {data_path}")

Using data from: projects\house-price-regression\data\kc_house_data.csv


In [3]:
df = pd.read_csv(data_path)

In [4]:
df.head()

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,7129300520,10/13/2014,221900.0,3,1.00,1180,5650,1.0,0,0,...,7,1180,0,1955,0,98178,47.5112,-122.257,1340,5650
1,6414100192,12/9/2014,538000.0,3,2.25,2570,7242,2.0,0,0,...,7,2170,400,1951,1991,98125,47.7210,-122.319,1690,7639
2,5631500400,2/25/2015,180000.0,2,1.00,770,10000,1.0,0,0,...,6,770,0,1933,0,98028,47.7379,-122.233,2720,8062
3,2487200875,12/9/2014,604000.0,4,3.00,1960,5000,1.0,0,0,...,7,1050,910,1965,0,98136,47.5208,-122.393,1360,5000
4,1954400510,2/18/2015,510000.0,3,2.00,1680,8080,1.0,0,0,...,8,1680,0,1987,0,98074,47.6168,-122.045,1800,7503


In [5]:
df.isnull().sum() # Check that if we have a null data
# False -> 0
# True -> 1

id               0
date             0
price            0
bedrooms         0
bathrooms        0
sqft_living      0
sqft_lot         0
floors           0
waterfront       0
view             0
condition        0
grade            0
sqft_above       0
sqft_basement    0
yr_built         0
yr_renovated     0
zipcode          0
lat              0
long             0
sqft_living15    0
sqft_lot15       0
dtype: int64

In [6]:
df.describe().transpose()

,count,mean,std,min,25%,50%,75%,max
id,21597.0,4.580474e+09,2.876736e+09,1.000102e+06,2.123049e+09,3.904930e+09,7.308900e+09,9.900000e+09
price,21597.0,5.402966e+05,3.673681e+05,7.800000e+04,3.220000e+05,4.500000e+05,6.450000e+05,7.700000e+06
bedrooms,21597.0,3.373200e+00,9.262989e-01,1.000000e+00,3.000000e+00,3.000000e+00,4.000000e+00,3.300000e+01
bathrooms,21597.0,2.115826e+00,7.689843e-01,5.000000e-01,1.750000e+00,2.250000e+00,2.500000e+00,8.000000e+00
sqft_living,21597.0,2.080322e+03,9.181061e+02,3.700000e+02,1.430000e+03,1.910000e+03,2.550000e+03,1.354000e+04
sqft_lot,21597.0,1.509941e+04,4.141264e+04,5.200000e+02,5.040000e+03,7.618000e+03,1.068500e+04,1.651359e+06
floors,21597.0,1.494096e+00,5.396828e-01,1.000000e+00,1.000000e+00,1.500000e+00,2.000000e+00,3.500000e+00
waterfront,21597.0,7.547345e-03,8.654900e-02,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,1.000000e+00
view,21597.0,2.342918e-01,7.663898e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,4.000000e+00
condition,21597.0,3.409825e+00,6.505456e-01,1.000000e+00,3.000000e+00,3.000000e+00,4.000000e+00,5.000000e+00


In [7]:
sns.displot(df['price'], kde=True)

In [8]:
# sns.countplot(df['bedrooms'])
# I've just tried to run it but it has problems and it wont' show the true result and will take time to execute

In [9]:
sns.histplot(df['bedrooms'])

<Axes: xlabel='price', ylabel='Count'>

In [10]:
df.corr(numeric_only= True)['price'].sort_values() # (correlate)Don't forget to set the numeric_only parameter as true

zipcode         -0.053402
id              -0.016772
long             0.022036
condition        0.036056
yr_built         0.053953
sqft_lot15       0.082845
sqft_lot         0.089876
yr_renovated     0.126424
floors           0.256804
waterfront       0.266398
lat              0.306692
bedrooms         0.308787
sqft_basement    0.323799
view             0.397370
bathrooms        0.525906
sqft_living15    0.585241
sqft_above       0.605368
grade            0.667951
sqft_living      0.701917
price            1.000000
Name: price, dtype: float64

In [11]:
plt.figure(figsize=(12,8))
sns.scatterplot(x = 'price', y = 'sqft_living', data= df)

<Axes: xlabel='price', ylabel='sqft_living'>

In [12]:
plt.figure(figsize=(10,5))
sns.boxplot(x='bedrooms',y='price',data = df)

<Axes: xlabel='bedrooms', ylabel='price'>

In [13]:
df.columns

Index(['id', 'date', 'price', 'bedrooms', 'bathrooms', 'sqft_living',
       'sqft_lot', 'floors', 'waterfront', 'view', 'condition', 'grade',
       'sqft_above', 'sqft_basement', 'yr_built', 'yr_renovated', 'zipcode',
       'lat', 'long', 'sqft_living15', 'sqft_lot15'],
      dtype='str')

In [14]:
plt.figure(figsize=(12,8))
sns.scatterplot(x='price',y='long',data=df)

<Axes: xlabel='price', ylabel='long'>

In [15]:
plt.figure(figsize=(12,8))
sns.scatterplot(x='price',y='lat',data=df)

<Axes: xlabel='price', ylabel='lat'>

In [16]:
plt.figure(figsize=(12,8))
sns.scatterplot(x='long',y='lat', data=df, hue='price')

<Axes: xlabel='long', ylabel='lat'>

In [17]:
df.sort_values('price', ascending=False).head(20)

,id,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
7245,6762700020,10/13/2014,7700000.0,6,8.00,12050,27600,2.5,0,3,...,13,8570,3480,1910,1987,98102,47.6298,-122.323,3940,8800
3910,9808700762,6/11/2014,7060000.0,5,4.50,10040,37325,2.0,1,2,...,11,7680,2360,1940,2001,98004,47.6500,-122.214,3930,25449
9245,9208900037,9/19/2014,6890000.0,6,7.75,9890,31374,2.0,0,4,...,13,8860,1030,2001,0,98039,47.6305,-122.240,4540,42730
4407,2470100110,8/4/2014,5570000.0,5,5.75,9200,35069,2.0,0,0,...,13,6200,3000,2001,0,98039,47.6289,-122.233,3560,24345
1446,8907500070,4/13/2015,5350000.0,5,5.00,8000,23985,2.0,0,4,...,12,6720,1280,2009,0,98004,47.6232,-122.220,4600,21750
1313,7558700030,4/13/2015,5300000.0,6,6.00,7390,24829,2.0,1,4,...,12,5000,2390,1991,0,98040,47.5631,-122.210,4320,24619
1162,1247600105,10/20/2014,5110000.0,5,5.25,8010,45517,2.0,1,4,...,12,5990,2020,1999,0,98033,47.6767,-122.211,3430,26788
8085,1924059029,6/17/2014,4670000.0,5,6.75,9640,13068,1.0,1,4,...,12,4820,4820,1983,2009,98040,47.5570,-122.210,3270,10454
2624,7738500731,8/15/2014,4500000.0,5,5.50,6640,40014,2.0,1,4,...,12,6350,290,2004,0,98155,47.7493,-122.280,3030,23408
8629,3835500195,6/18/2014,4490000.0,4,3.00,6430,27517,2.0,0,0,...,12,6430,0,2001,0,98004,47.6208,-122.219,3720,14592


In [18]:
int(len(df) * 0.01)

215

In [19]:
non_top_1_percent = df.sort_values('price', ascending=False).iloc[int(len(df) * 0.01):]

In [20]:
plt.figure(figsize=(12,8))
sns.scatterplot(x='long', y='lat', data=non_top_1_percent, hue='price',
                edgecolor=None, alpha=0.2, palette='RdYlGn')

<Axes: xlabel='long', ylabel='lat'>

In [21]:
sns.boxplot(x='waterfront', y='price', data=df)

<Axes: xlabel='long', ylabel='lat'>

In [22]:
df = df.drop('id', axis = 1)

In [23]:
df['date'] = pd.to_datetime(df['date']) # It would be more usefull than that string and we can really easy work on it

In [24]:
df['date']

0       2014-10-13
1       2014-12-09
2       2015-02-25
3       2014-12-09
4       2015-02-18
           ...    
21592   2014-05-21
21593   2015-02-23
21594   2014-06-23
21595   2015-01-16
21596   2014-10-15
Name: date, Length: 21597, dtype: datetime64[us]

In [25]:
df['year'] = df['date'].apply(lambda date: date.year)
df['month'] = df['date'].apply(lambda date: date.month)


In [26]:
df.head()

,date,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,...,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15,year,month
0,2014-10-13,221900.0,3,1.00,1180,5650,1.0,0,0,3,...,0,1955,0,98178,47.5112,-122.257,1340,5650,2014,10
1,2014-12-09,538000.0,3,2.25,2570,7242,2.0,0,0,3,...,400,1951,1991,98125,47.7210,-122.319,1690,7639,2014,12
2,2015-02-25,180000.0,2,1.00,770,10000,1.0,0,0,3,...,0,1933,0,98028,47.7379,-122.233,2720,8062,2015,2
3,2014-12-09,604000.0,4,3.00,1960,5000,1.0,0,0,5,...,910,1965,0,98136,47.5208,-122.393,1360,5000,2014,12
4,2015-02-18,510000.0,3,2.00,1680,8080,1.0,0,0,3,...,0,1987,0,98074,47.6168,-122.045,1800,7503,2015,2


In [27]:
plt.figure(figsize=(10,5))
sns.boxplot(x='month', y='price', data = df)

<Axes: xlabel='month', ylabel='price'>

In [28]:
df.groupby('month')['price'].mean().plot()

<Axes: xlabel='month', ylabel='price'>

In [29]:
df.groupby('year')['price'].mean().plot()

<Axes: xlabel='year', ylabel='price'>

In [30]:
df = df.drop('date', axis = 1)

In [31]:
# df['zipcode'].value_counts()

In [32]:
df = df.drop('zipcode', axis=1)
# I dropped zipcode here because it is just a label, not a real numeric scale.

In [33]:
df

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,year,month
0,221900.0,3,1.00,1180,5650,1.0,0,0,3,7,1180,0,1955,0,47.5112,-122.257,1340,5650,2014,10
1,538000.0,3,2.25,2570,7242,2.0,0,0,3,7,2170,400,1951,1991,47.7210,-122.319,1690,7639,2014,12
2,180000.0,2,1.00,770,10000,1.0,0,0,3,6,770,0,1933,0,47.7379,-122.233,2720,8062,2015,2
3,604000.0,4,3.00,1960,5000,1.0,0,0,5,7,1050,910,1965,0,47.5208,-122.393,1360,5000,2014,12
4,510000.0,3,2.00,1680,8080,1.0,0,0,3,8,1680,0,1987,0,47.6168,-122.045,1800,7503,2015,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21592,360000.0,3,2.50,1530,1131,3.0,0,0,3,8,1530,0,2009,0,47.6993,-122.346,1530,1509,2014,5
21593,400000.0,4,2.50,2310,5813,2.0,0,0,3,8,2310,0,2014,0,47.5107,-122.362,1830,7200,2015,2
21594,402101.0,2,0.75,1020,1350,2.0,0,0,3,7,1020,0,2009,0,47.5944,-122.299,1020,2007,2014,6
21595,400000.0,3,2.50,1600,2388,2.0,0,0,3,8,1600,0,2004,0,47.5345,-122.069,1410,1287,2015,1


In [34]:
X = df.drop('price', axis = 1).values # If write this values method it will return us the numpy array not the dataframe
y = df['price'].values

In [35]:
from sklearn.model_selection import train_test_split

In [36]:
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=101 )

In [37]:
from sklearn.preprocessing import MinMaxScaler

In [38]:
scaler = MinMaxScaler()

In [39]:
X_train = scaler.fit_transform(X_train)  # Fit on train only, then reuse it for test data.

In [40]:
X_test = scaler.transform(X_test)

In [41]:
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential

In [42]:
X_train.shape

(15117, 19)

## Model

First I train on the full dataset and keep an eye on the validation loss.

In [43]:
model = Sequential()

model.add(Dense(19, activation='relu'))
model.add(Dense(19, activation='relu'))
model.add(Dense(19, activation='relu'))
model.add(Dense(19, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')  # mse = mean squared error

In [44]:
early_stop = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    batch_size=128,
    epochs=300,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Stopped after {len(history.history['loss'])} epochs")

Stopped after 300 epochs


In [45]:
losses = pd.DataFrame(history.history)
losses.tail()

,loss,val_loss
295,3.031193e+10,2.827975e+10
296,3.030401e+10,2.826931e+10
297,3.029631e+10,2.825912e+10
298,3.028842e+10,2.824797e+10
299,3.028059e+10,2.823750e+10


In [46]:
losses.plot()
# I compare train loss and validation loss here.

<Axes: >

In [47]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, explained_variance_score

In [48]:
predictions = model.predict(X_test, verbose=0)

In [49]:
np.sqrt(mean_squared_error(y_test , predictions))

np.float64(168040.16857614586)

In [50]:
mean_absolute_error(y_test, predictions)

103976.91887357735

In [51]:
df['price'].describe()

count    2.159700e+04
mean     5.402966e+05
std      3.673681e+05
min      7.800000e+04
25%      3.220000e+05
50%      4.500000e+05
75%      6.450000e+05
max      7.700000e+06
Name: price, dtype: float64

In [52]:
explained_variance_score(y_test, predictions)  # Closer to 1 is better.

0.7871656955943485

In [53]:
plt.figure(figsize=(12,6))
plt.scatter(y_test, predictions)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r')

In [54]:
errors = y_test.reshape(-1, 1) - predictions

In [55]:
sns.displot(errors, kde=True)

In [56]:
single_house_features = df.drop('price', axis = 1).iloc[0]
len(single_house_features)

19

In [57]:
single_house = scaler.transform(single_house_features.values.reshape(1, -1))  # Keep it as one row for the model.

In [58]:
model.predict(single_house, verbose=0)

array([[290327.97]], dtype=float32)

In [59]:
df.head(1)

,price,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,condition,grade,sqft_above,sqft_basement,yr_built,yr_renovated,lat,long,sqft_living15,sqft_lot15,year,month
0,221900.0,3,1.0,1180,5650,1.0,0,0,3,7,1180,0,1955,0,47.5112,-122.257,1340,5650,2014,10


In [60]:
int(len(df) * 0.02)

431

## What I tried next

Then I dropped the top 2% most expensive houses to see if the outliers were making the regression harder.

In [61]:
ndf = df.sort_values('price', ascending=False).iloc[int(len(df) * 0.02):]

In [62]:
X = ndf.drop('price', axis = 1).values
y = ndf['price'].values

In [63]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

In [64]:
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [65]:
X_train.shape

(14816, 19)

In [66]:
model = Sequential()

model.add(Dense(19, activation='relu'))
model.add(Dense(38, activation='relu'))
model.add(Dense(19, activation='relu'))
model.add(Dense(38, activation='relu'))
model.add(Dense(19, activation='relu'))
model.add(Dense(38, activation='relu'))
model.add(Dense(1))

model.compile(optimizer='adam', loss='mse')

In [67]:
early_stop = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    batch_size=64,
    epochs=400,
    callbacks=[early_stop],
    verbose=0,
)

print(f"Stopped after {len(history.history['loss'])} epochs")

Stopped after 400 epochs


In [68]:
losses = pd.DataFrame(history.history)
losses.tail()

,loss,val_loss
395,1.067660e+10,1.098841e+10
396,1.066993e+10,1.098831e+10
397,1.066368e+10,1.097972e+10
398,1.065792e+10,1.097575e+10
399,1.065071e+10,1.097329e+10


In [69]:
plt.figure(figsize=(12,12))
losses.plot()
# This second run is just to compare the curve after removing the big outliers.

<Axes: >

In [70]:
predictions = model.predict(X_test, verbose=0)

In [71]:
np.sqrt(mean_squared_error(y_test, predictions))

np.float64(104753.46039180657)

In [72]:
mean_absolute_error(y_test, predictions)

70479.37631151575

In [73]:
ndf['price'].describe()

count    2.116600e+04
mean     5.056971e+05
std      2.567843e+05
min      7.800000e+04
25%      3.200000e+05
50%      4.460000e+05
75%      6.260000e+05
max      1.600000e+06
Name: price, dtype: float64

In [74]:
explained_variance_score(y_test, predictions)  # Closer to 1 is better.

0.8376194858798462

In [75]:
plt.figure(figsize=(12,6))
plt.scatter(y_test, predictions)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r')

In [76]:
single_house = scaler.transform(single_house_features.values.reshape(1, -1))  # Reuse the same raw row with the new scaler.

In [77]:
model.predict(single_house, verbose=0)

array([[251109.05]], dtype=float32)